# ESP32-P4 Gesture Model Training

This notebook trains a small 224x224 grayscale gesture classifier for later ESP-DL deployment.

Expected source layout:

```text
gesture_dataset_fisheye_224/
  a_little/
  good/
  help/
  me/
  name/
  sign_language/
  you/
```

The notebook will create `dataset_split/`, `outputs/`, and `exports/` under this folder.

In [1]:
from pathlib import Path
import json
import math
import random
import shutil
import time

import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path.cwd()
SOURCE_DIR = ROOT / "gesture_dataset_fisheye_224"
SPLIT_DIR = ROOT / "dataset_split"
OUTPUT_DIR = ROOT / "outputs"
EXPORT_DIR = ROOT / "exports"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 100
PATIENCE = 20
LR = 3e-3
WEIGHT_DECAY = 1e-4
TRAIN_RATIO = 0.75
VAL_RATIO = 0.15
TEST_RATIO = 0.10
RECREATE_SPLIT = True

OUTPUT_DIR.mkdir(exist_ok=True)
EXPORT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("root:", ROOT)
print("device:", device)

root: d:\esp32p4\example-standalone-inferencing-espressif-esp32-main\Mymodel
device: cuda


In [2]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Missing dataset folder: {SOURCE_DIR}")

all_class_dirs = sorted([p for p in SOURCE_DIR.iterdir() if p.is_dir()])
valid_files_by_class = {}
skipped_files = []

for class_dir in all_class_dirs:
    valid_files = []
    for path in sorted(class_dir.iterdir()):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:
            continue
        if not path.name.startswith(class_dir.name + "_"):
            skipped_files.append((path, "filename/folder label mismatch"))
            continue
        try:
            with Image.open(path) as image:
                if image.size != (IMG_SIZE, IMG_SIZE):
                    skipped_files.append((path, f"size {image.size}"))
                    continue
                image.verify()
        except Exception as exc:
            skipped_files.append((path, f"unreadable: {exc}"))
            continue
        valid_files.append(path)
    if valid_files:
        valid_files_by_class[class_dir.name] = valid_files

class_dirs = [p for p in all_class_dirs if p.name in valid_files_by_class]
class_names = [p.name for p in class_dirs]
if len(class_names) < 2:
    raise RuntimeError("At least two non-empty 224x224 class folders are required")

print("classes:", class_names)
total = 0
for class_dir in class_dirs:
    count = len(valid_files_by_class[class_dir.name])
    total += count
    print(f"{class_dir.name:16s} {count:5d}")
print("valid 224x224 images:", total)
print("skipped images:", len(skipped_files))
for path, reason in skipped_files:
    print(f"  skip {path.relative_to(SOURCE_DIR)}: {reason}")

classes: ['a_little', 'good', 'help', 'me', 'name', 'sign_language', 'you']
a_little            86
good               104
help               105
me                 118
name                68
sign_language       51
you                115
total images: 647


In [3]:
def split_one_class(files, train_ratio, val_ratio, seed):
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    n = len(files)
    n_train = max(1, int(round(n * train_ratio)))
    n_val = max(1, int(round(n * val_ratio))) if n >= 3 else 0
    if n_train + n_val >= n:
        n_train = max(1, n - 2) if n >= 3 else max(1, n - 1)
        n_val = 1 if n >= 3 else 0
    train_files = files[:n_train]
    val_files = files[n_train:n_train + n_val]
    test_files = files[n_train + n_val:]
    return train_files, val_files, test_files

if RECREATE_SPLIT and SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for subset in ["train", "val", "test"]:
    for class_name in class_names:
        (SPLIT_DIR / subset / class_name).mkdir(parents=True, exist_ok=True)

split_counts = {subset: {name: 0 for name in class_names} for subset in ["train", "val", "test"]}

for class_dir in class_dirs:
    files = valid_files_by_class[class_dir.name]
    train_files, val_files, test_files = split_one_class(files, TRAIN_RATIO, VAL_RATIO, SEED)
    for subset, subset_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
        for src in subset_files:
            dst = SPLIT_DIR / subset / class_dir.name / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
        split_counts[subset][class_dir.name] = len(subset_files)

for subset in ["train", "val", "test"]:
    print("\n", subset)
    for class_name in class_names:
        print(f"  {class_name:16s} {split_counts[subset][class_name]:5d}")


 train
  a_little            64
  good                78
  help                79
  me                  88
  name                51
  sign_language       38
  you                 86

 val
  a_little            13
  good                16
  help                16
  me                  18
  name                10
  sign_language        8
  you                 17

 test
  a_little             9
  good                10
  help                10
  me                  12
  name                 7
  sign_language        5
  you                 12


In [4]:
train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = datasets.ImageFolder(SPLIT_DIR / "train", transform=train_tfms)
val_ds = datasets.ImageFolder(SPLIT_DIR / "val", transform=eval_tfms)
test_ds = datasets.ImageFolder(SPLIT_DIR / "test", transform=eval_tfms)

assert train_ds.classes == class_names, (train_ds.classes, class_names)
num_classes = len(train_ds.classes)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

x, y = next(iter(train_loader))
print("classes:", train_ds.classes)
print("class_to_idx:", train_ds.class_to_idx)
print("sample batch:", x.shape, y.shape)

classes: ['a_little', 'good', 'help', 'me', 'name', 'sign_language', 'you']
class_to_idx: {'a_little': 0, 'good': 1, 'help': 2, 'me': 3, 'name': 4, 'sign_language': 5, 'you': 6}
sample batch: torch.Size([32, 1, 96, 96]) torch.Size([32])


In [5]:
class ConvBNAct(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, groups=1):
        padding = kernel_size // 2
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

class DSConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(in_ch, in_ch, kernel_size=3, stride=stride, groups=in_ch),
            ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1),
        )

    def forward(self, x):
        return self.block(x)

class TinyGestureCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNAct(1, 16, kernel_size=3, stride=2),    # 224 -> 112
            DSConv(16, 24, stride=1),
            DSConv(24, 32, stride=2),                     # 112 -> 56
            DSConv(32, 48, stride=1),
            DSConv(48, 64, stride=2),                     # 56 -> 28
            DSConv(64, 96, stride=1),
            DSConv(96, 128, stride=2),                    # 28 -> 14
            DSConv(128, 160, stride=1),
            DSConv(160, 192, stride=2),                   # 14 -> 7
            DSConv(192, 192, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=0.20)
        self.classifier = nn.Linear(192, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.classifier(x)

model = TinyGestureCNN(num_classes).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(model)
print("parameters:", param_count)

TinyGestureCNN(
  (features): Sequential(
    (0): ConvBNAct(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): DSConv(
      (block): Sequential(
        (0): ConvBNAct(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): ConvBNAct(
          (0): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
      )
    )
    (2): DSConv(
      (block): Sequential(
        (0): ConvBNAct(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=

In [6]:
def make_class_weights(dataset):
    counts = np.zeros(len(dataset.classes), dtype=np.float32)
    for _, label in dataset.samples:
        counts[label] += 1
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

class_weights = make_class_weights(train_ds).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8)

def run_epoch(model, loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [7]:
best_val_acc = -1.0
best_epoch = 0
bad_epochs = 0
history = []
best_path = OUTPUT_DIR / "gesture_cnn_224_best.pth"

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    scheduler.step(val_acc)
    lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": lr,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }
    history.append(row)
    print(f"epoch {epoch:03d} lr={lr:.2e} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        bad_epochs = 0
        torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": train_ds.class_to_idx,
            "img_size": IMG_SIZE,
            "epoch": epoch,
            "val_acc": val_acc,
        }, best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stop at epoch {epoch}, best epoch {best_epoch}, best val_acc {best_val_acc:.4f}")
            break

with open(OUTPUT_DIR / "history_224.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print("training seconds:", round(time.time() - start_time, 1))
print("best model:", best_path)

epoch 001 lr=1.00e-03 train_loss=1.9444 train_acc=0.1219 val_loss=1.9481 val_acc=0.1633
epoch 002 lr=1.00e-03 train_loss=1.8489 train_acc=0.2727 val_loss=1.9859 val_acc=0.2041
epoch 003 lr=1.00e-03 train_loss=1.8157 train_acc=0.2769 val_loss=1.8812 val_acc=0.2551
epoch 004 lr=1.00e-03 train_loss=1.7810 train_acc=0.3285 val_loss=1.8467 val_acc=0.2041
epoch 005 lr=1.00e-03 train_loss=1.7316 train_acc=0.3285 val_loss=1.8216 val_acc=0.2551
epoch 006 lr=1.00e-03 train_loss=1.6786 train_acc=0.3802 val_loss=1.7803 val_acc=0.3061
epoch 007 lr=1.00e-03 train_loss=1.6275 train_acc=0.4112 val_loss=1.7302 val_acc=0.3163
epoch 008 lr=1.00e-03 train_loss=1.5873 train_acc=0.4318 val_loss=1.6672 val_acc=0.3673
epoch 009 lr=1.00e-03 train_loss=1.4997 train_acc=0.4442 val_loss=1.6803 val_acc=0.3878
epoch 010 lr=1.00e-03 train_loss=1.4743 train_acc=0.4566 val_loss=1.5063 val_acc=0.4184
epoch 011 lr=1.00e-03 train_loss=1.4073 train_acc=0.4690 val_loss=1.5071 val_acc=0.4388
epoch 012 lr=1.00e-03 train_loss

In [8]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

test_loss, test_acc = run_epoch(model, test_loader, train=False)
print(f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}")

conf = torch.zeros(num_classes, num_classes, dtype=torch.int64)
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu()
        for true_label, pred_label in zip(labels, preds):
            conf[true_label, pred_label] += 1

print("confusion matrix rows=true cols=pred")
print("classes:", class_names)
print(conf.numpy())

test_loss=0.4134 test_acc=0.8923
confusion matrix rows=true cols=pred
classes: ['a_little', 'good', 'help', 'me', 'name', 'sign_language', 'you']
[[ 8  1  0  0  0  0  0]
 [ 0  8  0  0  1  1  0]
 [ 0  0 10  0  0  0  0]
 [ 0  0  0 10  1  0  1]
 [ 0  0  0  0  7  0  0]
 [ 0  0  0  0  0  5  0]
 [ 1  0  0  1  0  0 10]]


In [10]:
model_cpu = TinyGestureCNN(num_classes)
model_cpu.load_state_dict(ckpt["model_state"])
model_cpu.eval()

onnx_path = EXPORT_DIR / "gesture_cnn_224_gray.onnx"
dummy = torch.randn(1, 1, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model_cpu,
    dummy,
    onnx_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamo=False,
)

(EXPORT_DIR / "labels.txt").write_text("\n".join(class_names) + "\n", encoding="utf-8")
with open(EXPORT_DIR / "class_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(train_ds.class_to_idx, f, indent=2)
with open(EXPORT_DIR / "preprocess.json", "w", encoding="utf-8") as f:
    json.dump({
        "input_shape": [1, 1, IMG_SIZE, IMG_SIZE],
        "color": "grayscale",
        "resize": [IMG_SIZE, IMG_SIZE],
        "normalization": "x = (pixel / 255.0 - 0.5) / 0.5",
        "training_range": [-1.0, 1.0],
        "classes": class_names,
    }, f, indent=2)

print("exported:", onnx_path)
print("labels:", EXPORT_DIR / "labels.txt")

W0704 16:10:40.443000 40988 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0704 16:10:41.322000 40988 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `TinyGestureCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyGestureCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Users\10944\.conda\envs\pt\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\10944\.conda\envs\pt\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\10944\.conda\envs\pt\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "c:\Users\10944\.conda\envs\pt\Lib\site-packages\onnxscript\version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported: d:\esp32p4\example-standalone-inferencing-espressif-esp32-main\Mymodel\exports\gesture_cnn_96_gray.onnx
labels: d:\esp32p4\example-standalone-inferencing-espressif-esp32-main\Mymodel\exports\labels.txt


In [11]:
from PIL import Image

def predict_image(path):
    img = Image.open(path).convert("L")
    x = eval_tfms(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    order = probs.argsort()[::-1]
    return [(class_names[i], float(probs[i])) for i in order]

sample_path = test_ds.samples[0][0]
print("sample:", sample_path)
for label, prob in predict_image(sample_path):
    print(f"{label:16s} {prob:.4f}")

sample: d:\esp32p4\example-standalone-inferencing-espressif-esp32-main\Mymodel\dataset_split\test\a_little\a_little_000004_20260701_233823_367.jpg
a_little         0.8877
you              0.0410
sign_language    0.0372
name             0.0137
me               0.0109
good             0.0084
help             0.0011


## Next deployment step

Use `exports/gesture_cnn_224_gray.onnx` as the source model for ESP-PPQ quantization and ESP-DL conversion.

Keep the ESP32 preprocessing consistent with `preprocess.json`: grayscale 224x224 and normalization equivalent to `(pixel / 255 - 0.5) / 0.5` before quantization. During ESP-DL int8 conversion this normalization should be folded into the quantized input calibration path when possible.